# 2장 — 징검다리: 사슬의 수학 (핵심)

웹 챕터(2장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다:
단계마다 정확도가 다른 사슬의 곱, 병목 개선 실험, 그리고 로그 변환.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch2')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 사슬의 곱: 주간 보고서 다섯 단계

가상 업무 "주간 보고서": 자료 검색 0.99 · 수치 추출 0.95 · 표 만들기 0.90 · 요약 0.99 · **형식 맞추기 0.70**.
각 단계가 독립이면 일 전체의 성공률은 전부 **곱한 값**입니다:

$$P = p_1 \times p_2 \times \cdots \times p_n = \prod_{i=1}^{n} p_i$$

In [ ]:
import math

NAMES = ['자료 검색', '수치 추출', '표 만들기', '요약', '형식 맞추기']
BASE = [0.99, 0.95, 0.90, 0.99, 0.70]

def chain(ps):
    """사슬 전체의 성공률 = 각 단계 확률의 곱."""
    return math.prod(ps)

base = chain(BASE)
print(f'기본 사슬: {" × ".join(f"{p:.2f}" for p in BASE)} = {base:.5f} ≈ {base*100:.1f}%')
bottleneck = NAMES[BASE.index(min(BASE))]
print(f'병목 단계: {bottleneck} ({min(BASE):.2f})')

# 웹 실험실(ChainMathLab)과 동일한 수치인지 자가검증
assert abs(base - 0.58658985) < 1e-9
assert round(base * 100, 1) == 58.7
assert bottleneck == '형식 맞추기'
print('✅ 기본 사슬 58.7%, 웹과 일치')

## 2. 개선 실험: 같은 노력, 다른 효과

① 병목(0.70 → 0.90)만 개선 vs ② 이미 좋은 단계(0.99 → 0.999)를 개선.
그리고 보너스: ③ 병목 단계를 **아예 없애면**(자동화 도구로 대체) 어떻게 될까요?

In [ ]:
fix_bottleneck = chain([0.99, 0.95, 0.90, 0.99, 0.90])   # ① 병목만 0.90 으로
fix_good       = chain([0.999, 0.95, 0.90, 0.99, 0.70])  # ② 우등생을 0.999 로
remove_step    = chain([0.99, 0.95, 0.90, 0.99])         # ③ 병목 단계 제거 (4단계)

print(f'기본            : {base*100:.1f}%')
print(f'① 병목 개선     : {fix_bottleneck*100:.1f}%  (+{(fix_bottleneck-base)*100:.1f}%p)')
print(f'② 우등생 개선   : {fix_good*100:.1f}%  (+{(fix_good-base)*100:.1f}%p)')
print(f'③ 병목 제거     : {remove_step*100:.1f}%  (+{(remove_step-base)*100:.1f}%p). 안 곱하는 것이 최고의 개선')

assert abs(fix_bottleneck - 0.75418695) < 1e-9   # 75.4%
assert abs(fix_good - 0.591922485) < 1e-9        # 59.2%
assert abs(remove_step - 0.8379855) < 1e-9       # 83.8%
assert round((fix_bottleneck - base) * 100, 1) == 16.8  # 부동소수 반올림 기준 +16.8%p(표기상 16.7%p)
print('✅ 개선 실험 수치, 웹과 일치')

## 3. 로그 안경: 곱의 세계를 합의 세계로

$$\log P = \log p_1 + \log p_2 + \cdots + \log p_n$$

각 단계의 $-\log_{10} p_i$ 를 막대로 그리면 "누가 성공률을 갉아먹는지"가 한눈에 보입니다.
막대의 **합**이 전체의 $-\log_{10} P$ 가 되니까요(2부 7장의 log(a×b)=log a+log b).

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

nlogs = [-math.log10(p) for p in BASE]
assert abs(sum(nlogs) - (-math.log10(base))) < 1e-12  # 막대의 합 = 전체의 −log₁₀
ratio = nlogs[4] / nlogs[0]
print(f'−log₁₀ 기여: 형식 맞추기 {nlogs[4]:.4f} vs 자료 검색 {nlogs[0]:.4f}, {ratio:.0f}배')
assert round(ratio) == 35

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = ['#f43f5e' if p == min(BASE) else '#6366f1' for p in BASE]
bars = ax.barh(NAMES[::-1], nlogs[::-1], color=colors[::-1])
for b, v in zip(bars, nlogs[::-1]):
    ax.annotate(f'{v:.4f}', (v, b.get_y() + b.get_height() / 2), textcoords='offset points', xytext=(4, -4), fontsize=9)
ax.set_xlabel('−log₁₀ pᵢ (감쇠 기여: 클수록 성공률을 많이 깎음)')
ax.set_title(f'로그 안경으로 본 사슬: 막대의 합 {sum(nlogs):.4f} = 전체의 −log₁₀ ({base*100:.1f}%)')
plt.tight_layout()
plt.show()

In [ ]:
# 보너스 — 균일 사슬(모든 단계가 p)에서 성공률이 절반 밑으로 떨어지는 단계 수
# p^n < 0.5  ↔  n > log(0.5)/log(p)
for pct in [80, 90, 95, 99]:
    p = pct / 100
    n_half = math.ceil(math.log(0.5) / math.log(p))
    print(f'걸음 {pct}% → {n_half}단계부터 성공률 절반 미만')
assert math.ceil(math.log(0.5) / math.log(0.9)) == 7
assert math.ceil(math.log(0.5) / math.log(0.99)) == 69  # 99%짜리 일꾼도 69단계면 절반
print('✅ 어떤 일꾼도 사슬이 충분히 길면 무너진다. 되살리기(4~6장)가 필요한 이유')

## 4. 오늘의 발견

- **일의 성공률 = 각 단계 확률의 곱** (P = Πpᵢ): 기본 사슬 58.7%의 범인은 병목(0.70) 하나.
- **개선은 병목부터**: 병목 개선 +16.7%p vs 우등생 개선 +0.5%p (33배 차이),
  그리고 **단계 제거(83.8%)가 개선(75.4%)보다 낫다**. 안 곱하는 것이 최고의 개선.
- **로그 변환**이 곱을 합으로 펴서 감쇠 기여를 비교하게 해 준다 (−log 막대 = 우선순위표).
- 99% 일꾼도 69단계면 절반 아래. 긴 사슬은 되살리기 없이 못 버틴다.

다음 장(3장 도구를 드는 손)에서 AI 일꾼이 언제 검색·계산기를 집어 드는지 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**